# Session1_Task10 — Churn Analysis

## สิ่งที่ต้องทำ
1. ระบุลูกค้าที่ churn จาก `customers_cleaned.csv`
2. คำนวณ churn rate (%) = churned ÷ ลูกค้าทั้งหมด × 100
3. คำนวณ avg CLTV แยก churned vs active

**หมายเหตุ:** คอลัมน์ `churned` มีหลายรูปแบบ (True/False/Yes/No/1/0/Y/N) ต้อง normalize ก่อน

In [ ]:
import pandas as pd
import warnings; warnings.filterwarnings('ignore')

c = pd.read_csv('customers_cleaned.csv')
cltv_df = pd.read_csv('Session1_CLTV.csv')  # โหลด CLTV จาก Task 9

print('churned unique values:', c['churned'].unique())

In [ ]:
# --- 1. Normalize churned column ---

# ค่าที่ถือว่า churned = True
churned_true = {'true', 'yes', '1', 'y'}

# .str.lower() → แปลงเป็นตัวพิมพ์เล็กทั้งหมด
# .str.strip() → ลบช่องว่างหน้า-หลัง
# .isin()      → เช็คว่าอยู่ใน set ที่กำหนดไหม
c['is_churned'] = c['churned'].astype(str).str.lower().str.strip().isin(churned_true)

print('Churned distribution:')
print(c['is_churned'].value_counts())

In [ ]:
# --- 2. Churn Rate ---

# churn_rate = จำนวนที่ churn / ทั้งหมด × 100
churn_rate = round(c['is_churned'].sum() / len(c) * 100, 2)
print(f'Churn rate: {churn_rate}%')

In [ ]:
# --- 3. Average CLTV: Churned vs Active ---

# merge customer กับ CLTV
# how='inner' → เก็บแค่ลูกค้าที่มีทั้งใน customers และ CLTV
merged = c[['customer_id','is_churned']].merge(cltv_df, on='customer_id', how='inner')

# .groupby().mean() → ค่าเฉลี่ย CLTV แยกตาม is_churned
avg_cltv = merged.groupby('is_churned')['cltv'].mean().round(2)

avg_cltv_churned = float(avg_cltv.get(True,  0))
avg_cltv_active  = float(avg_cltv.get(False, 0))

print(f'Avg CLTV churned: {avg_cltv_churned}')
print(f'Avg CLTV active : {avg_cltv_active}')

# --- Export ---
result = pd.DataFrame([{
    'churn_rate'      : churn_rate,
    'avg_cltv_churned': avg_cltv_churned,
    'avg_cltv_active' : avg_cltv_active
}])

display(result)
result.to_csv('Session1_Churn_Analysis.csv', index=False)
print('✅ Saved Session1_Churn_Analysis.csv')

# === จุดสังเกต ===
# ✔ ไฟล์มี 1 แถว 3 คอลัมน์
# ✔ churn_rate อยู่ระหว่าง 0-100
# ✔ avg_cltv_active มักสูงกว่า avg_cltv_churned (ลูกค้าที่ยังอยู่มี CLTV สูงกว่า)